# 5 — Convergence

**The concept:** an iterative block needs to decide when to stop. SmartMDAO never makes you infer
that from a list of residuals — every iterative block hands back a **`ConvergenceReport`** saying
what happened and why.

There are three possible endings: `CONVERGED`, `MAX_ITERATIONS`, `ABANDONED`.

In [1]:
from smartmdao import (
    Pipeline, IterativeSolver, ConvergenceReport,
    CONVERGED, MAX_ITERATIONS, ABANDONED,
)

print("the three statuses:", CONVERGED, "|", MAX_ITERATIONS, "|", ABANDONED)

the three statuses: converged | max_iterations | abandoned


In [2]:
settled = Pipeline(solver=IterativeSolver(tolerance=1e-9, target_var="x"))

@settled.step(outputs=["x"])
def relax(x: float) -> float:
    return 0.5 * x + 1.0

result = settled.run(x=0.0)
report = result["convergence_reports"][-1]

print("status:    ", report.status)
print("converged: ", report.converged)
print("iterations:", report.iterations)
print("steps:     ", report.steps)
print("x ->", round(result["x"], 9))
print()
print("is a ConvergenceReport:", isinstance(report, ConvergenceReport))

status:     converged
converged:  True
iterations: 31
steps:      ('relax',)
x -> 1.999999999

is a ConvergenceReport: True


## Running out of iterations is not converging

The distinction matters and is easy to miss if you only look at the final value. A run that hit its
iteration cap says so.

In [3]:
capped = Pipeline(solver=IterativeSolver(tolerance=1e-12, max_iterations=3, target_var="x"))

@capped.step(outputs=["x"])
def relax_slowly(x: float) -> float:
    return 0.5 * x + 1.0

out = capped.run(x=0.0)
capped_report = out["convergence_reports"][-1]

print("status:   ", capped_report.status)
print("converged:", capped_report.converged)
print("residuals:", [round(r, 6) for r in capped_report.residuals])
print("x ->", round(out["x"], 6), "(the true fixed point is 2.0)")

Reached max_iterations (3) without converging. Last residual: 2.500000e-01


status:    max_iterations
converged: False
residuals: [1.0, 0.5, 0.25]
x -> 1.75 (the true fixed point is 2.0)


## `target_var` — converge on one variable, not on everything

By default the residual is a `max()` across **every** variable a block produces. One noisy variable
can then hold the whole system back, or mask a variable that has not settled.

`target_var` narrows convergence to a single variable. It is also **required** whenever you use a
stateful checker, because a stateful checker needs `distance()` called exactly once per iteration.

In [4]:
from smartmdao import HybridSolver

# The README's own hand-written loop converges on y2 ALONE. To translate that
# faithfully you must say so - otherwise you get a different criterion.
faithful = Pipeline(solver=HybridSolver(target_var="y2", tolerance=1e-6))

@faithful.step(outputs=["y1"])
def discipline_1(z: float, y2: float) -> float:
    return z**2 - 0.2 * y2

@faithful.step(outputs=["y2"])
def discipline_2(y1: float) -> float:
    return abs(y1) ** 0.5

faithful_out = faithful.run(z=2.0, y2=1.0)
print("watching y2 only:", faithful_out["convergence_reports"][0].iterations, "iterations")

idiomatic = Pipeline(solver=HybridSolver(tolerance=1e-6))
for fn, outs in ((discipline_1, ["y1"]), (discipline_2, ["y2"])):
    idiomatic.add(fn, outputs=outs)
idiomatic_out = idiomatic.run(z=2.0, y2=1.0)
print("watching everything:", idiomatic_out["convergence_reports"][0].iterations, "iterations")

watching y2 only: 6 iterations
watching everything: 7 iterations


## Oscillation — stopping a loop that will never settle

Some couplings do not converge; they flip between two states forever. A tolerance check cannot
detect that, because the residual never shrinks and never grows.

`OscillationAwareConvergenceChecker` remembers the history and recognises a repeat.

In [5]:
from smartmdao import OscillationAwareConvergenceChecker

flip = Pipeline(
    solver=HybridSolver(
        max_iterations=100,
        target_var="choice",
        convergence_checker=OscillationAwareConvergenceChecker(),
    )
)

@flip.step(outputs=["choice"])
def flip_flop(choice: str, nudge: int) -> str:
    return "B" if choice == "A" else "A"

@flip.step(outputs=["nudge"])
def observe(choice: str) -> int:
    return len(choice)

flip_out = flip.run(choice="A", nudge=0)
flip_report = flip_out["convergence_reports"][-1]

print("status:    ", flip_report.status)
print("iterations:", flip_report.iterations, "(of a possible 100)")
print("reason:    ", flip_report.reason)

Abandoned at iteration 4: coupling variable is oscillating with period 2, cycling between ['B', 'A']; it will never satisfy the convergence tolerance


status:     abandoned
iterations: 4 (of a possible 100)
reason:     coupling variable is oscillating with period 2, cycling between ['B', 'A']; it will never satisfy the convergence tolerance


It stopped at iteration 4 instead of burning all 100, **and it kept the trace**. That trace is
the point: an abandoned solve tells you *why* it failed, not merely that it did.

## `AbandonmentAware` — how that works

Rather than widening the `ConvergenceChecker` protocol (which would have broken every existing
checker), abandonment is a **second, optional protocol**. A checker that implements
`abandon_reason()` can stop a solve; one that does not is never asked.

In [6]:
from smartmdao import AbandonmentAware, ConvergenceChecker, StandardConvergenceChecker

standard = StandardConvergenceChecker()
oscillation = OscillationAwareConvergenceChecker()

print("StandardConvergenceChecker    is AbandonmentAware:", isinstance(standard, AbandonmentAware))
print("OscillationAware...Checker     is AbandonmentAware:", isinstance(oscillation, AbandonmentAware))
print()
print("both satisfy ConvergenceChecker:",
      isinstance(standard, ConvergenceChecker), isinstance(oscillation, ConvergenceChecker))

StandardConvergenceChecker    is AbandonmentAware: False
OscillationAware...Checker     is AbandonmentAware: True

both satisfy ConvergenceChecker: True True


Writing your own takes one method. Structural typing means no inheritance and no registration.

In [7]:
class GiveUpAfterThree:
    """Converges normally, but refuses to iterate more than three times."""

    def __init__(self):
        self.seen = 0

    def distance(self, previous, current) -> float:
        self.seen += 1
        return abs(current - previous) if previous is not None else float("inf")

    def abandon_reason(self):
        return "three iterations is my limit" if self.seen > 3 else None

impatient = Pipeline(
    solver=IterativeSolver(max_iterations=50, target_var="x",
                           convergence_checker=GiveUpAfterThree())
)

@impatient.step(outputs=["x"])
def crawl(x: float) -> float:
    return 0.999 * x + 1.0

impatient_report = impatient.run(x=0.0)["convergence_reports"][-1]
print("status:", impatient_report.status)
print("reason:", impatient_report.reason)
print("is AbandonmentAware:", isinstance(GiveUpAfterThree(), AbandonmentAware))

Abandoned at iteration 4: three iterations is my limit


status: abandoned
reason: three iterations is my limit
is AbandonmentAware: True


## Raising instead of reporting

`OscillationDetectedError` is still available if you would rather have an exception — pass
`raise_on_detection=True`. The trade is that you **lose the residual history**, which is what
prompted the default to change.

In [8]:
from smartmdao import OscillationDetectedError

raising = Pipeline(
    solver=IterativeSolver(
        max_iterations=100,
        target_var="choice",
        convergence_checker=OscillationAwareConvergenceChecker(raise_on_detection=True),
    )
)

@raising.step(outputs=["choice"])
def flip_flop_2(choice: str) -> str:
    return "B" if choice == "A" else "A"

try:
    raising.run(choice="A")
except OscillationDetectedError as error:
    print(f"OscillationDetectedError: {error}")

Pipeline execution failed: Coupling variable is oscillating with period 2 (detected at iteration 4); it cycles between: ['B', 'A']. This will never satisfy the convergence tolerance.


OscillationDetectedError: Coupling variable is oscillating with period 2 (detected at iteration 4); it cycles between: ['B', 'A']. This will never satisfy the convergence tolerance.


---

**Next:** [6 — Non-numeric convergence](06-non-numeric-convergence.ipynb), where the thing being
converged is not a number at all.